## Bonus Challenge Four: Processing Streaming Data

### Install dependencies

In [1]:
# google-cloud-bigquery and google-cloud-pubsub provide the BigQuery + Pub/Sub clients.
%pip install --quiet --upgrade google-cloud-bigquery google-cloud-pubsub db-dtypes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 7.7 MB/s eta 0:00:00


In [2]:
from google.cloud import bigquery
from google.cloud import pubsub_v1

### Variables

In [3]:
import os

# --- Diagnostic: shows what each detection method returns ---
print("=== Project detection diagnostic ===")
print("env GOOGLE_CLOUD_PROJECT:", os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("env GCP_PROJECT         :", os.environ.get("GCP_PROJECT"))
try:
    import google.auth
    _creds, _proj = google.auth.default()
    print("google.auth project     :", _proj)
except Exception as e:
    print("google.auth failed      :", e)
print("=" * 36)


def detect_project_id():
    # 1. Explicit env var wins (lets anyone override without editing code)
    env = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT")
    if env:
        return env
    # 2. Ask Application Default Credentials what project we're running under
    try:
        import google.auth
        _, project = google.auth.default()
        if project:
            return project
    except Exception:
        pass
    # 3. Last resort: query the metadata server (works on GCP runtimes)
    try:
        import urllib.request
        req = urllib.request.Request(
            "http://metadata.google.internal/computeMetadata/v1/project/project-id",
            headers={"Metadata-Flavor": "Google"},
        )
        return urllib.request.urlopen(req, timeout=2).read().decode()
    except Exception:
        return None


class Config:
    """Immutable run config. `project_id` is YOUR billing project (auto-detected).
    The Pub/Sub topic lives in a DIFFERENT project (paul-leroy) that we only subscribe to."""

    def __init__(
        self,
        project_id=None,                       # None -> auto-detect your billing project
        dataset="transponder_data_colab",
        location="US",
        table_name="flight_transponder_msgs",
        # --- Pub/Sub source (provided by the challenge) ---
        pubsub_project="paul-leroy",
        topic_name="flight-transponder",
        subscription_name=None,                # None -> auto-generate a unique pull subscription
    ):
        resolved = project_id or detect_project_id()
        if not resolved:
            raise ValueError(
                "Could not determine project_id. Pass it explicitly: "
                "Config(project_id='my-project')"
            )
        import uuid as _uuid
        sub = subscription_name or f"flight-transponder-sub-{_uuid.uuid4().hex[:8]}"
        # bypass our own __setattr__ block during construction
        for k, v in {
            "project_id": resolved, "dataset": dataset, "location": location,
            "table_name": table_name, "pubsub_project": pubsub_project,
            "topic_name": topic_name, "subscription_name": sub,
        }.items():
            object.__setattr__(self, k, v)

    def __setattr__(self, name, value):
        raise AttributeError(f"Config is immutable; can't set {name!r}")

    @property
    def table(self):
        return f"{self.project_id}.{self.dataset}.{self.table_name}"

    @property
    def topic_path(self):
        return f"projects/{self.pubsub_project}/topics/{self.topic_name}"

    @property
    def subscription_path(self):
        # Subscription is created in YOUR project, attached to the source topic.
        return f"projects/{self.project_id}/subscriptions/{self.subscription_name}"


CFG = Config()

print("\nProject      :", CFG.project_id)
print("Table        :", CFG.table)
print("Source topic :", CFG.topic_path)
print("Subscription :", CFG.subscription_path)

=== Project detection diagnostic ===
env GOOGLE_CLOUD_PROJECT: qwiklabs-gcp-01-5fe45b5e4e14
env GCP_PROJECT         : None
google.auth project     : qwiklabs-gcp-01-5fe45b5e4e14

Project      : qwiklabs-gcp-01-5fe45b5e4e14
Table        : qwiklabs-gcp-01-5fe45b5e4e14.transponder_data_colab.flight_transponder_msgs
Source topic : projects/paul-leroy/topics/flight-transponder
Subscription : projects/qwiklabs-gcp-01-5fe45b5e4e14/subscriptions/flight-transponder-sub-e64e3ef0


### Logging

In [4]:
import logging, json, sys, uuid, datetime as dt

logger = logging.getLogger("flight_transponder_stream")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

RUN_ID = str(uuid.uuid4())

def log_event(step: str, status: str, **kw):
    logger.info(json.dumps({"run_id": RUN_ID, "step": step, "status": status, **kw}))

log_event("init", "ok", started=dt.datetime.now(dt.timezone.utc).isoformat())

2026-06-02 19:49:33,147 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "init", "status": "ok", "started": "2026-06-02T19:49:33.147928+00:00"}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "init", "status": "ok", "started": "2026-06-02T19:49:33.147928+00:00"}


### Pipeline: Pub/Sub -> parse -> BigQuery

In [5]:
import time


class TransponderStreamPipeline:
    """Creates a BigQuery table for SBS transponder messages, pulls the streaming
    CSV feed from a Pub/Sub topic, parses each message into the 22-field schema,
    and stream-inserts the rows into BigQuery for querying and Geo Viz."""

    # 22-field SBS BaseStation schema. Order matches the CSV stream exactly so we
    # can zip the split CSV fields straight onto these columns.
    SCHEMA = [
        bigquery.SchemaField("MT",   "STRING",  mode="NULLABLE", description="SEL ID AIR STA CLK MSG info http://woodair.net/sbs/Article/Barebones42_Socket_Data.htm"),
        bigquery.SchemaField("TT",   "INT64",   mode="NULLABLE", description="1 - 8"),
        bigquery.SchemaField("SID",  "STRING",  mode="NULLABLE", description="Database Session record number"),
        bigquery.SchemaField("AID",  "STRING",  mode="NULLABLE", description="Database Aircraft record number"),
        bigquery.SchemaField("Hex",  "STRING",  mode="NULLABLE", description="Aircraft Mode S hexadecimal code https://opensky-network.org/datasets/metadata/"),
        bigquery.SchemaField("FID",  "STRING",  mode="NULLABLE", description="Database Flight record number"),
        bigquery.SchemaField("DMG",  "DATE",    mode="NULLABLE", description="Date message generated"),
        bigquery.SchemaField("TMG",  "TIME",    mode="NULLABLE", description="Time message generated"),
        bigquery.SchemaField("DML",  "DATE",    mode="NULLABLE", description="Date message logged"),
        bigquery.SchemaField("TML",  "TIME",    mode="NULLABLE", description="Time message logged"),
        bigquery.SchemaField("CS",   "STRING",  mode="NULLABLE", description="Callsign (flight number or registration)"),
        bigquery.SchemaField("Alt",  "INT64",   mode="NULLABLE", description="Mode C altitude (Flight Level)"),
        bigquery.SchemaField("GS",   "INT64",   mode="NULLABLE", description="Ground Speed"),
        bigquery.SchemaField("Trk",  "INT64",   mode="NULLABLE", description="Track"),
        bigquery.SchemaField("Lat",  "FLOAT64", mode="NULLABLE", description="Latitude (N/E positive, S/W negative)"),
        bigquery.SchemaField("Lng",  "FLOAT64", mode="NULLABLE", description="Longitude (N/E positive, S/W negative)"),
        bigquery.SchemaField("VR",   "INT64",   mode="NULLABLE", description="Vertical Rate"),
        bigquery.SchemaField("Sq",   "STRING",  mode="NULLABLE", description="Assigned Mode A squawk code"),
        bigquery.SchemaField("Alrt", "INT64",   mode="NULLABLE", description="Flag to indicate squawk has changed"),
        bigquery.SchemaField("Emer", "INT64",   mode="NULLABLE", description="Flag to indicate emergency code has been set"),
        bigquery.SchemaField("SPI",  "INT64",   mode="NULLABLE", description="Flag to indicate transponder Ident has been activated"),
        bigquery.SchemaField("Gnd",  "INT64",   mode="NULLABLE", description="Flag to indicate ground squat switch is active"),
    ]

    # Column name -> BigQuery type, used to coerce each CSV field to the right Python type.
    FIELD_TYPES = [(f.name, f.field_type) for f in SCHEMA]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.client = bigquery.Client(project=cfg.project_id, location=cfg.location)
        self.subscriber = pubsub_v1.SubscriberClient()
        self.rows_inserted = 0
        self.messages_seen = 0
        self.parse_errors = 0

    # ---- BigQuery setup -------------------------------------------------
    def ensure_dataset(self):
        ds = bigquery.Dataset(f"{self.cfg.project_id}.{self.cfg.dataset}")
        ds.location = self.cfg.location
        self.client.create_dataset(ds, exists_ok=True)
        log_event("dataset_ready", "ok", dataset=self.cfg.dataset)

    def ensure_table(self):
        table = bigquery.Table(self.cfg.table, schema=self.SCHEMA)
        self.client.create_table(table, exists_ok=True)
        log_event("table_ready", "ok", table=self.cfg.table, fields=len(self.SCHEMA))

    # ---- Pub/Sub setup --------------------------------------------------
    def ensure_subscription(self):
        """Create a pull subscription in our project attached to the source topic."""
        try:
            self.subscriber.create_subscription(
                request={
                    "name": self.cfg.subscription_path,
                    "topic": self.cfg.topic_path,
                    "ack_deadline_seconds": 30,
                }
            )
            log_event("subscription_created", "ok", sub=self.cfg.subscription_path)
        except Exception as e:
            # AlreadyExists is fine; re-raise anything else (e.g. permissions).
            if "AlreadyExists" in type(e).__name__ or "already exists" in str(e).lower():
                log_event("subscription_created", "exists", sub=self.cfg.subscription_path)
            else:
                raise

    # ---- Parsing --------------------------------------------------------
    @staticmethod
    def _coerce(value, bq_type):
        """Convert a raw CSV field to the Python type BigQuery expects.
        Empty strings -> None (the feed leaves most fields blank)."""
        if value is None:
            return None
        value = value.strip()
        if value == "":
            return None
        try:
            if bq_type == "INT64":
                return int(value)
            if bq_type == "FLOAT64":
                return float(value)
            if bq_type == "DATE":
                # feed uses YYYY/MM/DD -> ISO YYYY-MM-DD
                return value.replace("/", "-")
            if bq_type == "TIME":
                # HH:MM:SS.mmm is already valid BigQuery TIME
                return value
            return value  # STRING
        except (ValueError, TypeError):
            return None

    def parse_message(self, raw: str):
        """Turn one CSV transponder line into a dict keyed by schema column names.
        Returns None if the field count doesn't match the 22-field schema."""
        fields = raw.strip().split(",")
        if len(fields) != len(self.FIELD_TYPES):
            return None
        return {
            name: self._coerce(fields[i], bq_type)
            for i, (name, bq_type) in enumerate(self.FIELD_TYPES)
        }

    # ---- Streaming loop -------------------------------------------------
    def collect(self, run_seconds: int = 180, batch_size: int = 200,
                max_messages_per_pull: int = 100):
        """Pull from the subscription for `run_seconds`, parsing and stream-inserting
        rows into BigQuery in batches. Then stop. (Step 4: run a few minutes, then stop.)"""
        deadline = time.time() + run_seconds
        buffer = []
        log_event("collect_start", "ok", run_seconds=run_seconds)

        while time.time() < deadline:
            resp = self.subscriber.pull(
                request={
                    "subscription": self.cfg.subscription_path,
                    "max_messages": max_messages_per_pull,
                },
                timeout=30,
            )
            if not resp.received_messages:
                continue

            ack_ids = []
            for rm in resp.received_messages:
                ack_ids.append(rm.ack_id)
                self.messages_seen += 1
                raw = rm.message.data.decode("utf-8", errors="replace")
                # A single Pub/Sub message may carry one or several CSV lines.
                for line in raw.splitlines():
                    if not line.strip():
                        continue
                    row = self.parse_message(line)
                    if row is None:
                        self.parse_errors += 1
                        continue
                    buffer.append(row)

            # Acknowledge so messages aren't redelivered.
            self.subscriber.acknowledge(
                request={"subscription": self.cfg.subscription_path, "ack_ids": ack_ids}
            )

            if len(buffer) >= batch_size:
                self._flush(buffer)
                buffer = []

        if buffer:
            self._flush(buffer)

        log_event("collect_done", "ok", messages_seen=self.messages_seen,
                  rows_inserted=self.rows_inserted, parse_errors=self.parse_errors)

    def _flush(self, rows):
        errors = self.client.insert_rows_json(self.cfg.table, rows)
        if errors:
            log_event("flush", "error", count=len(errors), sample=errors[:2])
        else:
            self.rows_inserted += len(rows)
            log_event("flush", "ok", inserted=len(rows), total=self.rows_inserted)

    # ---- Queries --------------------------------------------------------
    def count_records(self):
        df = self.client.query(
            f"SELECT COUNT(*) AS record_count FROM `{self.cfg.table}`"
        ).to_dataframe()
        log_event("count_records", "ok", count=int(df.iloc[0]["record_count"]))
        return df

    def preview(self, n: int = 10):
        df = self.client.query(
            f"SELECT * FROM `{self.cfg.table}` LIMIT {n}"
        ).to_dataframe()
        log_event("preview", "ok", rows=len(df))
        return df

    def locations(self):
        """Step 6: return ST_GEOGPOINT locations for every message with coordinates.
        This is the query to open in Geo Viz."""
        sql = f"""
        SELECT
          ST_GEOGPOINT(Lng, Lat) AS Location
        FROM `{self.cfg.table}`
        WHERE Lat IS NOT NULL
          AND Lng IS NOT NULL
        """
        df = self.client.query(sql).to_dataframe()
        log_event("locations", "ok", points=len(df))
        return df

    def cleanup_subscription(self):
        """Optional: delete the temporary pull subscription when done."""
        try:
            self.subscriber.delete_subscription(
                request={"subscription": self.cfg.subscription_path}
            )
            log_event("cleanup_subscription", "ok")
        except Exception as e:
            log_event("cleanup_subscription", "skip", reason=str(e))

### Setup: dataset + table

In [6]:
pipeline = TransponderStreamPipeline(CFG)
pipeline.ensure_dataset()
pipeline.ensure_table()
print(f"Table ready: {CFG.table}")

2026-06-02 19:50:38,461 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "dataset_ready", "status": "ok", "dataset": "transponder_data_colab"}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "dataset_ready", "status": "ok", "dataset": "transponder_data_colab"}


2026-06-02 19:50:38,829 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "table_ready", "status": "ok", "table": "qwiklabs-gcp-01-5fe45b5e4e14.transponder_data_colab.flight_transponder_msgs", "fields": 22}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "table_ready", "status": "ok", "table": "qwiklabs-gcp-01-5fe45b5e4e14.transponder_data_colab.flight_transponder_msgs", "fields": 22}


Table ready: qwiklabs-gcp-01-5fe45b5e4e14.transponder_data_colab.flight_transponder_msgs


### Subscribe to the Pub/Sub feed

In [7]:
pipeline.ensure_subscription()
print(f"Subscribed: {CFG.subscription_path}")
print(f"   -> topic: {CFG.topic_path}")

2026-06-02 19:51:10,759 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "subscription_created", "status": "ok", "sub": "projects/qwiklabs-gcp-01-5fe45b5e4e14/subscriptions/flight-transponder-sub-e64e3ef0"}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "subscription_created", "status": "ok", "sub": "projects/qwiklabs-gcp-01-5fe45b5e4e14/subscriptions/flight-transponder-sub-e64e3ef0"}


Subscribed: projects/qwiklabs-gcp-01-5fe45b5e4e14/subscriptions/flight-transponder-sub-e64e3ef0
   -> topic: projects/paul-leroy/topics/flight-transponder


### Collect streaming data — run a few minutes, then stop

This pulls messages, parses each CSV line into the 22-field schema, and stream-inserts
batches into BigQuery. Adjust `run_seconds` as needed (default 3 minutes).

In [8]:
pipeline.collect(run_seconds=180)
print(f"Messages seen : {pipeline.messages_seen:,}")
print(f"Rows inserted : {pipeline.rows_inserted:,}")
print(f"Parse errors  : {pipeline.parse_errors:,}")

2026-06-02 19:51:38,428 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "collect_start", "status": "ok", "run_seconds": 180}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "collect_start", "status": "ok", "run_seconds": 180}


2026-06-02 19:51:43,165 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "flush", "status": "ok", "inserted": 200, "total": 200}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "flush", "status": "ok", "inserted": 200, "total": 200}


2026-06-02 19:51:45,504 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "flush", "status": "ok", "inserted": 200, "total": 400}


...

2026-06-02 19:54:38,539 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "flush", "status": "ok", "inserted": 15, "total": 50185}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "flush", "status": "ok", "inserted": 15, "total": 50185}


2026-06-02 19:54:38,541 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "collect_done", "status": "ok", "messages_seen": 50185, "rows_inserted": 50185, "parse_errors": 0}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "collect_done", "status": "ok", "messages_seen": 50185, "rows_inserted": 50185, "parse_errors": 0}


Messages seen : 50,185
Rows inserted : 50,185
Parse errors  : 0


### Preview the accumulated data

In [9]:
pipeline.preview()

2026-06-02 19:58:15,219 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "preview", "status": "ok", "rows": 10}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "preview", "status": "ok", "rows": 10}


,MT,TT,SID,AID,Hex,FID,DMG,TMG,DML,TML,...,GS,Trk,Lat,Lng,VR,Sq,Alrt,Emer,SPI,Gnd
0,MSG,7,1,1,400846,1,2026-06-02,19:51:10.439000,2026-06-02,19:51:10.487000,...,<NA>,<NA>,NaN,NaN,<NA>,None,<NA>,<NA>,<NA>,<NA>
1,MSG,7,1,1,010207,1,2026-06-02,19:51:10.445000,2026-06-02,19:51:10.487000,...,<NA>,<NA>,NaN,NaN,<NA>,None,<NA>,<NA>,<NA>,<NA>
2,MSG,4,1,1,4CAD7A,1,2026-06-02,19:51:10.450000,2026-06-02,19:51:10.487000,...,330,294,NaN,NaN,3200,None,<NA>,<NA>,<NA>,<NA>
3,MSG,3,1,1,3C5428,1,2026-06-02,19:51:10.451000,2026-06-02,19:51:10.488000,...,<NA>,<NA>,50.89468,0.71777,<NA>,None,0,<NA>,0,0
4,MSG,5,1,1,39E698,1,2026-06-02,19:51:10.451000,2026-06-02,19:51:10.488000,...,<NA>,<NA>,NaN,NaN,<NA>,None,0,<NA>,0,<NA>
5,MSG,7,1,1,4077E0,1,2026-06-02,19:51:10.452000,2026-06-02,19:51:10.488000,...,<NA>,<NA>,NaN,NaN,<NA>,None,<NA>,<NA>,<NA>,<NA>
6,MSG,5,1,1,40631B,1,2026-06-02,19:51:10.461000,2026-06-02,19:51:10.488000,...,<NA>,<NA>,NaN,NaN,<NA>,None,0,<NA>,0,<NA>
7,MSG,5,1,1,406CD2,1,2026-06-02,19:51:10.462000,2026-06-02,19:51:10.488000,...,<NA>,<NA>,NaN,NaN,<NA>,None,0,<NA>,0,<NA>
8,MSG,8,1,1,405B66,1,2026-06-02,19:51:10.470000,2026-06-02,19:51:10.488000,...,<NA>,<NA>,NaN,NaN,<NA>,None,<NA>,<NA>,<NA>,<NA>
9,MSG,7,1,1,406CD2,1,2026-06-02,19:51:10.476000,2026-06-02,19:51:10.489000,...,<NA>,<NA>,NaN,NaN,<NA>,None,<NA>,<NA>,<NA>,<NA>


### Step 5 — Count the records

In [10]:
pipeline.count_records()

2026-06-02 19:58:36,913 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "count_records", "status": "ok", "count": 50185}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "count_records", "status": "ok", "count": 50185}


,record_count
0,50185


### Step 6 — Locations query for Geo Viz

In [11]:
locations_sql = f"""
SELECT
  ST_GEOGPOINT(Lng, Lat) AS Location
FROM `{CFG.table}`
WHERE Lat IS NOT NULL
  AND Lng IS NOT NULL
"""
print(locations_sql)

# Also pull the points into a dataframe so we can confirm they look right.
pipeline.locations().head()


SELECT
  ST_GEOGPOINT(Lng, Lat) AS Location
FROM `qwiklabs-gcp-01-5fe45b5e4e14.transponder_data_colab.flight_transponder_msgs`
WHERE Lat IS NOT NULL
  AND Lng IS NOT NULL

2026-06-02 19:59:05,266 | INFO | {"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "locations", "status": "ok", "points": 2924}


INFO:flight_transponder_stream:{"run_id": "043eeb1b-473c-4ca9-afa5-c1a1c36c796f", "step": "locations", "status": "ok", "points": 2924}


,Location
0,POINT(0.71777 50.89468)
1,POINT(-0.3042 51.78621)
2,POINT(0.23087 51.64431)
3,POINT(-1.33573 51.50281)
4,POINT(-0.16068 51.28539)
